In [21]:
import os
import json
import math
import re
import sqlite3
import time
from collections import Counter, defaultdict
from typing import Any, TypedDict

import numpy as np
# from openai import OpenAI
import google.generativeai as genai


In [22]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY before running this notebook: export GEMINI_API_KEY=sk-...")

genai.configure(api_key=GEMINI_API_KEY)

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CHAT_MODEL = os.getenv("GEMINI_CHAT_MODEL", "gemini-3.1-flash-lite")
client = genai.GenerativeModel(model_name=CHAT_MODEL)

print(f"Gemini ready. Chat model: {CHAT_MODEL}; embedding model: {EMBED_MODEL}")

Gemini ready. Chat model: gemini-3.1-flash-lite; embedding model: BAAI/bge-small-en-v1.5


In [23]:
CORPUS = [
    {
        "id": "pods",
        "source": "pods.md",
        "text": "A Kubernetes Pod is the smallest deployable unit. Containers in a Pod share network, storage volumes, and lifecycle. Use kubectl describe pod and kubectl logs to debug Pod behavior.",
    },
    {
        "id": "deployment",
        "source": "deployment.md",
        "text": "A Deployment manages ReplicaSets and rolling updates. Use kubectl rollout status deployment/nginx to watch progress and kubectl rollout undo deployment/nginx to roll back a bad release.",
    },
    {
        "id": "service",
        "source": "service.md",
        "text": "A Service gives stable networking for Pods. ClusterIP is internal, NodePort exposes a port on nodes, and LoadBalancer asks the cloud provider for an external endpoint.",
    },
    {
        "id": "probes",
        "source": "probes.md",
        "text": "Readiness probes decide when a Pod can receive traffic. Liveness probes restart stuck containers. Startup probes protect slow-starting applications from premature restarts.",
    },
    {
        "id": "secrets",
        "source": "secrets.md",
        "text": "Kubernetes Secrets store sensitive values such as passwords, tokens, and keys. Enable encryption at rest, restrict RBAC, and avoid printing secret data in logs.",
    },
    {
        "id": "taints",
        "source": "taints.md",
        "text": "Taints repel Pods from nodes. Tolerations allow selected Pods to schedule onto tainted nodes. A NoSchedule taint blocks Pods that lack a matching toleration.",
    },
    {
        "id": "hpa",
        "source": "hpa.md",
        "text": "The HorizontalPodAutoscaler increases or decreases replicas based on CPU, memory, or custom metrics so an application can handle more traffic automatically.",
    },
]

NOISE_DOCS = [
    {"id": "noise-cache", "source": "cache-paper.md", "text": "Cache-aware matrix multiplication improves locality in CPU memory hierarchies and reduces cache misses."},
    {"id": "noise-graph", "source": "graph-paper.md", "text": "Graph partitioning algorithms optimize edge cuts in distributed computation workloads."},
]

print(f"Inline corpus loaded: {len(CORPUS)} K8s snippets + {len(NOISE_DOCS)} noise snippets")

Inline corpus loaded: 7 K8s snippets + 2 noise snippets


# Dense Search

In [24]:
# from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

# _embeddings = FastEmbedEmbeddings(model_name=EMBED_MODEL)

# def embed_texts(texts: list[str]) -> list[list[float]]:
#     response = _embeddings.embed_documents(texts)
#     return [item.embedding for item in response.data]

from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

_embeddings = FastEmbedEmbeddings(model_name=EMBED_MODEL)  # or EMBED_MODEL

def embed_texts(texts: list[str]) -> list[list[float]]:
    return _embeddings.embed_documents(texts)

In [25]:
def cosine(a: list[float], b: list[float]) -> float:
    av = np.array(a, dtype=float)
    bv = np.array(b, dtype=float)
    
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom else 0.0

In [26]:
def dense_search(question: str, docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    vectors = embed_texts([question] + [d["text"] for d in docs])
    qv, doc_vectors = vectors[0], vectors[1:]
    ranked = []
    for doc, dv in zip(docs, doc_vectors):
        ranked.append({**doc, "score": cosine(qv, dv)})
    return sorted(ranked, key=lambda d: d["score"], reverse=True)[:top_k]

In [27]:
def answer_with_context(question: str, chunks: list[dict[str, Any]]) -> str:
    context = "\n\n".join(f"SOURCE: {c['source']}\n{c['text']}" for c in chunks)

    prompt = f"""Answer only from the provided Kubernetes context. Cite source names inline.

Context:
{context}

Question: {question}"""

    response = client.generate_content(
        prompt,
        generation_config={"temperature": 0}
    )

    return response.text

# Sparse Search

In [28]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9_:-]+", text.lower())

In [29]:
def sparse_score(question: str, doc: dict[str, str]) -> float:
    q_terms = Counter(tokenize(question))
    d_terms = Counter(tokenize(doc["text"]))
    return sum(q_terms[t] * d_terms[t] for t in q_terms)

This function computes a **sparse (lexical) relevance score** — basically "how much do the question and document overlap word-by-word?" It's the classic bag-of-words matching used in keyword search (like a simplified BM25).

## Line by line

```python
def sparse_score(question: str, doc: dict[str, str]) -> float:
    q_terms = Counter(tokenize(question))      # word -> count for the question
    d_terms = Counter(tokenize(doc["text"]))   # word -> count for the document
    return sum(q_terms[t] * d_terms[t] for t in q_terms)  # dot product
```

1. **`tokenize(...)`** — splits text into words (e.g., lowercased, punctuation removed).
2. **`Counter(...)`** — counts how often each word appears.
3. **The sum** — multiplies the two counts for every word in the *question* and adds them up. This is exactly a **dot product of two sparse vectors** (hence the name: most dimensions are zero).

## Simple example

```python
question = "cat sat on mat"

doc = {"text": "the cat sat on the red mat"}
```

**Step 1 — count words in the question:**

| word | q_terms count |
|------|---------------|
| cat  | 1 |
| sat  | 1 |
| on   | 1 |
| mat  | 1 |

**Step 2 — count words in the document:**

| word | d_terms count |
|------|---------------|
| the  | 2 |
| cat  | 1 |
| sat  | 1 |
| on   | 1 |
| red  | 1 |
| mat  | 1 |

**Step 3 — dot product over the question's words only:**

```text
cat → 1 × 1 = 1
sat → 1 × 1 = 1
on  → 1 × 1 = 1
mat → 1 × 1 = 1
------------------
score = 4
```

("the" and "red" don't matter because they're not in the question.)

If the question repeated a word, it would count more:

```python
question = "cat cat sat"          # q_terms = {cat: 2, sat: 1}
score = 2×1 + 1×1 = 3             # repeated terms boost the score
```

And an unrelated doc scores **0**:

```python
doc2 = {"text": "dogs love bones"}   # no overlapping words
sparse_score(question, doc2)         # → 0.0
```

## Why "sparse"?

Each text is represented as a huge vector with one slot per vocabulary word, where almost all slots are `0`. Only a handful of words are nonzero — that's a *sparse* vector. This contrasts with **dense retrieval**, which uses embeddings (small vectors like 384 floats, all nonzero) that capture *meaning* ("car" ≈ "automobile") rather than exact words.

## In hybrid search

- **Sparse score** catches exact keywords ("error code E404") that embeddings might miss.
- **Dense score** catches paraphrases ("puppy" matches "dog") that exact matching misses.
- You combine both (e.g., weighted sum or reciprocal rank fusion) to get the best of both worlds.

⚠️ Note this simple version has known weaknesses real sparse methods fix: no IDF weighting (common words like "the" count as much as rare ones), no length normalization (longer docs win), and no fuzzy matching (typos/synonyms score 0).

In [30]:
def sparse_search(question: str, docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    ranked = [{**doc, "score": sparse_score(question, doc)} for doc in docs]
    return sorted(ranked, key=lambda d: d["score"], reverse=True)[:top_k]

In [31]:
def rrf_fuse(result_lists: list[list[dict[str, Any]]], k: int = 60, top_k: int = 3) -> list[dict[str, Any]]:
    scores: dict[str, float] = defaultdict(float)
    docs: dict[str, dict[str, Any]] = {}
    for results in result_lists:
        for rank, doc in enumerate(results, start=1):
            scores[doc["id"]] += 1 / (k + rank)
            docs[doc["id"]] = doc
    fused = [{**docs[doc_id], "rrf_score": score} for doc_id, score in scores.items()]
    return sorted(fused, key=lambda d: d["rrf_score"], reverse=True)[:top_k]

# Reciprocal Rank Fusion (RRF) Explained

This function implements **RRF**, the standard technique for merging ranked lists from *multiple retrieval systems* (e.g., BM25 keyword search + vector/semantic search) into a single ranking. It's called "hybrid search" because you combine lexical and semantic matches.

## Why not just merge scores?

BM25 returns unbounded scores (e.g., 12.7), while vector search returns cosine similarity (0–1). They're not comparable. RRF sidesteps this entirely — **it only uses ranks, never raw scores.**

## How it works

For every list, a document at rank `r` contributes:

```
contribution = 1 / (k + r)      # k = 60 (default), r starts at 1
```

Contributions are summed across all lists. Docs appearing high in *multiple* lists bubble to the top. The `k=60` constant **dampens** the gap between ranks, so position #1 doesn't overwhelmingly dominate position #10.

## Worked Example

Say you search `"payment failed"` and get these results:

```python
bm25_results = [
    {"id": "d1", "text": "Payment failed error codes"},
    {"id": "d3", "text": "Retry logic for payments"},
    {"id": "d5", "text": "Payment gateway setup"},
]
vector_results = [
    {"id": "d2", "text": "Troubleshooting transaction declines"},
    {"id": "d1", "text": "Payment failed error codes"},
    {"id": "d4", "text": "Refund policies"},
]

rrf_fuse([bm25_results, vector_results], k=60, top_k=3)
```

Score accumulation:

| Doc | From BM25 (rank → points) | From Vector (rank → points) | Total |
|-----|---------------------------|------------------------------|-------|
| **d1** | rank 1 → 1/61 = 0.01639 | rank 2 → 1/62 = 0.01613 | **0.03253** ✅ |
| **d2** | — | rank 1 → 1/61 = 0.01639 | **0.01639** |
| **d3** | rank 2 → 1/62 = 0.01613 | — | **0.01613** |
| d5 | rank 3 → 1/63 = 0.01587 | — | 0.01587 |
| d4 | — | rank 3 → 1/63 = 0.01587 | 0.01587 |

Output:

```python
[
    {"id": "d1", "text": "Payment failed error codes",        "rrf_score": 0.03252},
    {"id": "d2", "text": "Troubleshooting transaction declines", "rrf_score": 0.01639},
    {"id": "d3", "text": "Retry logic for payments",          "rrf_score": 0.01613},
]
```

Notice **d1 wins decisively** because both systems agree on it — that's exactly the signal RRF amplifies.

## Line-by-line breakdown

```python
scores[doc["id"]] += 1 / (k + rank)   # accumulate rank-based points per list
docs[doc["id"]] = doc                 # remember the doc payload (last list wins for metadata)
fused = [{**docs[doc_id], "rrf_score": score} ...]   # attach computed score to each doc
sorted(..., key=..., reverse=True)[:top_k]           # best-first, keep top 3
```

## Practical notes

- **`defaultdict(float)`**: lets you add to scores for unseen IDs without initializing them.
- **Ties** (like d4/d5 above): Python's `sorted` is stable, so earlier-seen docs come first.
- **Tuning `k`**: smaller `k` (e.g., 10) makes exact rank matter more; larger `k` flattens differences. 60 is the standard from the original Cormack et al. (2009) paper.
- **Metadata overwrite**: if the same ID appears in multiple lists with different payloads, the *last* list processed wins for the doc fields (only the score is summed).

In [32]:
exact_query = "kubectl rollout undo deployment/nginx"
print("Sparse results")
for c in sparse_search(exact_query, CORPUS + NOISE_DOCS):
    print(c["source"], c["score"])

print("\nDense results")
for c in dense_search(exact_query, CORPUS + NOISE_DOCS):
    print(c["source"], f"{c['score']:.3f}")

Sparse results
deployment.md 10
pods.md 2
service.md 0

Dense results
deployment.md 0.831
pods.md 0.576
taints.md 0.562


In [33]:
hybrid_query = "How do I send traffic to Pods from inside the cluster?"
dense = dense_search(hybrid_query, CORPUS + NOISE_DOCS, top_k=5)
sparse = sparse_search(hybrid_query, CORPUS + NOISE_DOCS, top_k=5)
fused = rrf_fuse([dense, sparse], top_k=5)

In [34]:
fused

[{'id': 'taints',
  'source': 'taints.md',
  'text': 'Taints repel Pods from nodes. Tolerations allow selected Pods to schedule onto tainted nodes. A NoSchedule taint blocks Pods that lack a matching toleration.',
  'score': 5,
  'rrf_score': 0.032266458495966696},
 {'id': 'service',
  'source': 'service.md',
  'text': 'A Service gives stable networking for Pods. ClusterIP is internal, NodePort exposes a port on nodes, and LoadBalancer asks the cloud provider for an external endpoint.',
  'score': 2,
  'rrf_score': 0.032018442622950824},
 {'id': 'probes',
  'source': 'probes.md',
  'text': 'Readiness probes decide when a Pod can receive traffic. Liveness probes restart stuck containers. Startup probes protect slow-starting applications from premature restarts.',
  'score': 2,
  'rrf_score': 0.0315136476426799},
 {'id': 'pods',
  'source': 'pods.md',
  'text': 'A Kubernetes Pod is the smallest deployable unit. Containers in a Pod share network, storage volumes, and lifecycle. Use kubect